# Managed Tables
Starter notebook for working with managed tables in Databricks.

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel — 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

spark.sql("SELECT current_catalog(), current_schema()").show(truncate=False)

In [ ]:
import os, sys
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel — 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

path = "/Volumes/population_metrics/landing/datasets/countries_dataset/countries_dataset/csv_data/countries_population/"
schema = StructType([
    StructField("COUNTRY_ID", IntegerType(), True),
    StructField("NAME", StringType(), True),
    StructField("NATIONALITY", StringType(), True),
    StructField("CODE", StringType(), True),
    StructField("ISO_ALPHA3", StringType(), True),
    StructField("CAPITAL", StringType(), True),
    StructField("POPULATION", IntegerType(), True),
    StructField("AREA_KM2", DoubleType(), True),
    StructField("REGION_ID", IntegerType(), True),
    StructField("SUB_REGION_ID", IntegerType(), True)
])

catalog_name = spark.sql("SELECT current_catalog() AS c").collect()[0]["c"]
schema_name = spark.sql("SELECT current_schema() AS s").collect()[0]["s"]
table_name = "country_population"
full_table_name = f"`{catalog_name}`.`{schema_name}`.`{table_name}`"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema_name}`")

df = spark.read.format("csv").options(header="true").schema(schema).load(path)
(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(full_table_name)
)

spark.sql(f"SELECT COUNT(*) AS row_count FROM {full_table_name}").show(truncate=False)
spark.table(full_table_name).show(20, truncate=False)
print(f"Saved managed table: {full_table_name}")

In [2]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel — 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

query = """
SELECT *
FROM dbw_dataengineering_sandbox_euwest_001.default.country_population
LIMIT 50
"""

spark.sql(query).show(truncate=False)

+----------+-----------------------------+-------------------------+----+----------+-------------------------+----------+---------+---------+-------------+
|COUNTRY_ID|NAME                         |NATIONALITY              |CODE|ISO_ALPHA3|CAPITAL                  |POPULATION|AREA_KM2 |REGION_ID|SUB_REGION_ID|
+----------+-----------------------------+-------------------------+----+----------+-------------------------+----------+---------+---------+-------------+
|210       |Sri Lanka                    |Sri Lankan               |LKA |LK        |Sri Jayawardenapura Kotte|21323733  |65610.0  |30       |30           |
|211       |Sudan                        |Sudanese                 |SDN |SD        |Khartoum                 |42813238  |1886068.0|50       |40           |
|212       |Suriname                     |Surinamese               |SUR |SR        |Paramaribo               |581372    |163820.0 |10       |10           |
|213       |Svalbard and Jan Mayen       |Svalbard              